In [18]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
import string
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity



# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
data = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv")
data.to_csv("submission.csv", index = False)

In [3]:
train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
train

,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A
...,...,...,...,...,...,...,...,...
1995,1996,What is the piezoelectric strain coefficient f...,d = 1.9·10‑12 m/V,d = 3.1·10‑12 m/V,d = 4.2·10‑12 m/V,d = 2.5·10‑12 m/V,d = 5.8·10‑12 m/V,B
1996,1997,Identify the correct statement: What is the sy...,A device used to demonstrate a neuro-inspired ...,A device used to demonstrate a neuro-inspired ...,A device used to demonstrate a neuro-inspired ...,A device used to demonstrate a neuro-inspired ...,A device used to demonstrate a neuro-inspired ...,E
1997,1998,Determine the correct option: What does Earnsh...,A collection of point charges can be maintaine...,A collection of point charges can be maintaine...,A collection of point charges can be maintaine...,A collection of point charges cannot be mainta...,A collection of point charges can be maintaine...,D
1998,1999,Identify the correct statement: What is the re...,The atmosphere is a mechanism that is only inf...,"The atmosphere possesses both chaos and order,...",The atmosphere is a structure that is only inf...,The atmosphere is a completely chaotic mechani...,The atmosphere is a completely ordered structu...,B


In [4]:
test = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")
test

,id,prompt,A,B,C,D,E
0,1,Pick the best possible answer: What is the rel...,"For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p..."
1,2,"What is the estimated redshift of CEERS-93316,...","Approximately z = 6.0, corresponding to 1 bill...","Approximately z = 16.7, corresponding to 235.8...","Approximately z = 3.0, corresponding to 5 bill...","Approximately z = 10.0, corresponding to 13 bi...","Approximately z = 13.0, corresponding to 30 bi..."
2,3,Pick the best possible answer: What is the rea...,The sun appears yellowish due to a reflection ...,"The longer wavelengths of light, such as red a...",The sun appears yellowish due to the scatterin...,The sun emits a yellow light due to its own sp...,The atmosphere absorbs the shorter wavelengths...
3,4,What is the significance of the redshift-dista...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...
4,5,What is the Landau-Lifshitz-Gilbert equation u...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...
...,...,...,...,...,...,...,...
495,496,What are the constituents of cold dark matter?,"They are unknown, but possibilities include la...",They are known to be black holes and Preon stars.,They are only MACHOs.,They are clusters of brown dwarfs.,They are new particles such as RAMBOs.
496,497,Pick the best possible answer: What is a plane...,A framework of planets that are all located in...,A mechanism of planets that are all the same s...,Any set of gravitationally bound non-stellar o...,A mechanism of planets that are all located in...,A structure of planets that are all made of gas.
497,498,Pick the best possible answer: What is magneti...,Magnetic susceptibility is a measure of how mu...,Magnetic susceptibility is a measure of how mu...,Magnetic susceptibility is a measure of how mu...,Magnetic susceptibility is a measure of how mu...,Magnetic susceptibility is a measure of how mu...
498,499,Determine the correct option: What is the evid...,The Milky Way galaxy has a supermassive black ...,The Milky Way galaxy has a supermassive black ...,The Milky Way galaxy has a supermassive black ...,The Milky Way galaxy has a supermassive black ...,The star S2 follows an elliptical orbit with a...


In [5]:
print(train.shape)
print(test.shape)

print(train.columns)

(2000, 8)
(500, 7)
Index(['id', 'prompt', 'A', 'B', 'C', 'D', 'E', 'answer'], dtype='object')


In [6]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 8 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   id      2000 non-null   int64 
 1   prompt  2000 non-null   object
 2   A       2000 non-null   object
 3   B       2000 non-null   object
 4   C       2000 non-null   object
 5   D       2000 non-null   object
 6   E       2000 non-null   object
 7   answer  2000 non-null   object
dtypes: int64(1), object(7)
memory usage: 125.1+ KB


In [7]:
test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 7 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   id      500 non-null    int64 
 1   prompt  500 non-null    object
 2   A       500 non-null    object
 3   B       500 non-null    object
 4   C       500 non-null    object
 5   D       500 non-null    object
 6   E       500 non-null    object
dtypes: int64(1), object(6)
memory usage: 27.5+ KB


Milestone 1 Question 1

Calculate the frequency distribution of the correct  answer  (A, B, C, D, E) in train.csv. Based on your counts, what is the sum of the occurrences of the most frequent option and the least frequent option?  

In [9]:
# Milestone 1 Q1
freq = train["answer"].value_counts()

print(freq)

answer_q1 = freq.max() + freq.min()

print("M1Q1 Answer =", answer_q1)

answer
B    490
C    459
A    369
D    358
E    324
Name: count, dtype: int64
M1Q1 Answer = 814


Milestone 1 Question 2

After converting the prompt column to lowercase and removing all standard punctuation characters (using Python's string.punctuation), split the text by whitespace. What is the total number of unique words (vocabulary size) across the entire cleaned prompt column of train.csv?  

In [10]:
# Milestone 1 Q2

all_prompts = " ".join(
    train["prompt"].astype(str)
)

cleaned = (
    all_prompts
    .lower()
    .translate(
        str.maketrans(
            "",
            "",
            string.punctuation
        )
    )
)

words = cleaned.split()

vocab_size = len(set(words))

print("M1Q2 Answer =", vocab_size)

M1Q2 Answer = 859


Milestone 1 Question 3

Using the cleaned prompt from Row ID 1, filter out the standard English stop words using sklearn.feature_extraction.text.ENGLISH_STOP_WORDS. How many words are left in the prompt for Row ID 1 after filtering?  

In [16]:
# Milestone 1 Q3
prompt = train.loc[0, "prompt"]

cleaned = (
    prompt.lower()
    .translate(
        str.maketrans(
            "",
            "",
            string.punctuation
        )
    )
)

tokens = cleaned.split()

filtered_tokens = [
    word
    for word in tokens
    if word not in ENGLISH_STOP_WORDS
]

print(filtered_tokens)
print("M1Q3 Answer =", len(filtered_tokens))

['pick', 'best', 'possible', 'answer', 'martin', 'heideggers', 'view', 'relationship', 'time', 'human', 'existence', 'listed', 'options']
M1Q3 Answer = 13


Milestone 1 Question 4 

Fit a default TfidfVectorizer(stop_words='english') on a list containing all the combined text of the prompts and options in train.csv. What is the exact total number of feature columns (vocabulary size) generated by the vectorizer?  

In [17]:
# Milestone 1 Q4
combined_text = []

for _, row in train.iterrows():

    text = (
        str(row["prompt"]) + " "
        + str(row["A"]) + " "
        + str(row["B"]) + " "
        + str(row["C"]) + " "
        + str(row["D"]) + " "
        + str(row["E"])
    )

    combined_text.append(text)

vectorizer = TfidfVectorizer(
    stop_words="english"
)

X = vectorizer.fit_transform(
    combined_text
)

print("M1Q4 Answer =", X.shape[1])

M1Q4 Answer = 2762


Milestone 1 Question 5

Using the TF-IDF vectorizer fitted in Question 3, calculate the cosine similarity between the prompt and option A strictly for Row ID 1. What is the resulting similarity score? (Round to 4 decimal places). 

In [21]:
# Milestone 1 Q5
prompt = train.loc[0, "prompt"]

option_a = train.loc[0, "A"]

prompt_vec = vectorizer.transform(
    [prompt]
)

option_vec = vectorizer.transform(
    [option_a]
)

similarity = cosine_similarity(
    prompt_vec,
    option_vec
)[0][0]

print(
    "M1Q5 Answer =",
    round(similarity, 4)
)

M1Q5 Answer = 0.272


Milestone 1 Question 6

Expand the logic from Question 4: For every row in train.csv, calculate the cosine similarity between the prompt and each of its 5 options .  Then calculate the percentage of instances where the option with the highest cosine similarity matches the correct answer.  

In [22]:
# Milestone 1 Q6

correct = 0

for _, row in train.iterrows():

    prompt = row["prompt"]

    options = {
        "A": row["A"],
        "B": row["B"],
        "C": row["C"],
        "D": row["D"],
        "E": row["E"]
    }

    prompt_vec = vectorizer.transform(
        [prompt]
    )

    similarities = {}

    for option_name, option_text in options.items():

        option_vec = vectorizer.transform(
            [option_text]
        )

        sim = cosine_similarity(
            prompt_vec,
            option_vec
        )[0][0]

        similarities[option_name] = sim

    prediction = max(
        similarities,
        key=similarities.get
    )

    if prediction == row["answer"]:
        correct += 1

accuracy = (
    correct
    / len(train)
) * 100

print(
    "Q6 Answer =",
    round(accuracy, 2)
)

Q6 Answer = 13.55


Milestone 1 Question 7

If the ground truth answer for a question is C, what is the MAP@3 score if a model predicts C A B ?  


ANSWER: 
Ground Truth Answer: C

Predicted Ranking: [C, A, B]

The correct answer appears at Rank 1.

For a single relevant answer, Average Precision at 3 (AP@3) is calculated as:
[
AP@3 = 1/ Rank
]

Therefore,
[
AP@3 = 1/1 = 1.0
]

Final Answer: 1.0

Milestone 1 Question 8

If the ground truth answer for a question is  B, what is the MAP@3 score if a model predicts D B E?  


Answer:

Ground Truth Answer: **B**


Predicted Ranking: **[D, B, E]**


The correct answer appears at **Rank 2**.


For a single relevant answer, Average Precision at 3 (AP@3) is calculated as:


[
AP@3 = 1/Rank
]


Therefore,

[
AP@3 = 1/2 = 0.5
]


**Final Answer: 0.5**


Milestone 1 Question 9 

The Majority Class Baseline: Find the most frequent correct answer in the training set (using your data from Q1). Make a static prediction for every single row where that most frequent answer is your 1st guess, followed by the second most frequent, and then the third most frequent. What is the overall MAP@3 score of this "Majority Class" baseline on train.csv?

In [23]:
# Milestone 1 Q9

freq = train["answer"].value_counts()

print(freq)

# Top 3 most frequent answers
top3 = list(freq.index[:3])

print("Top 3 answers:", top3)

def map3(actual, preds):
    for i, p in enumerate(preds):
        if p == actual:
            return 1/(i+1)
    return 0

scores = []

for ans in train["answer"]:
    scores.append(map3(ans, top3))

baseline_map3 = sum(scores) / len(scores)

print("MAP@3 =", baseline_map3)
print("Rounded =", round(baseline_map3, 4))

answer
B    490
C    459
A    369
D    358
E    324
Name: count, dtype: int64
Top 3 answers: ['B', 'C', 'A']
MAP@3 = 0.42125
Rounded = 0.4213


Milestone 1 Question 10

The TF-IDF Pipeline: Build a basic pipeline that evaluates every row in train.csv. For each row, calculate the TF-IDF cosine similarity between the prompt and each of the 5 options. Sort these options from highest similarity to lowest to form your top 3 predictions. What is the final average MAP@3 score of this TF-IDF pipeline across the entire training set? 

In [24]:
# Milestone 1 Q10

combined_text = (
    train["prompt"] + " " +
    train["A"] + " " +
    train["B"] + " " +
    train["C"] + " " +
    train["D"] + " " +
    train["E"]
)

vectorizer = TfidfVectorizer(stop_words="english")
vectorizer.fit(combined_text)

TfidfVectorizer(stop_words='english')

In [25]:
# Milestone 1 Q10

def map3(actual, preds):
    for i, p in enumerate(preds):
        if p == actual:
            return 1/(i+1)
    return 0

scores = []

for _, row in train.iterrows():

    prompt_vec = vectorizer.transform([row["prompt"]])

    similarities = {}

    for option in ["A", "B", "C", "D", "E"]:

        option_vec = vectorizer.transform([row[option]])

        sim = cosine_similarity(
            prompt_vec,
            option_vec
        )[0][0]

        similarities[option] = sim

    ranked_options = sorted(
        similarities,
        key=similarities.get,
        reverse=True
    )

    top3 = ranked_options[:3]

    score = map3(
        row["answer"],
        top3
    )

    scores.append(score)

final_map3 = sum(scores) / len(scores)

print("MAP@3 =", final_map3)
print("Rounded =", round(final_map3, 4))

MAP@3 = 0.2961666666666667
Rounded = 0.2962
